In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit import ClassicalRegister
import math

sim = BasicSimulator()

## Quantum Random Number Generation

Each random bit is obtained by preparing $|{+}\rangle = H|0\rangle$ and measuring. The outcome is 0 or 1 with equal probability using quantum source of randomness used by all three parties.

In [2]:
# ── Shared utility: quantum random bit generation ──────────────────────────────

def quantum_random_bits(n: int) -> list[int]:
    """
    Generate n random bits by measuring n qubits each prepared in |+> = H|0>.
    Generated in batches of 20 to respect the simulator qubit limit.
    Used by Alice, Eve, and Bob for all random choices.
    """
    bits = []
    batch_size = 20
    while len(bits) < n:
        size = min(batch_size, n - len(bits))
        qc = QuantumCircuit(size, size)
        for i in range(size):
            qc.h(i)          # Prepare |+> = (|0> + |1>) / sqrt(2)
        qc.measure(range(size), range(size))
        t = transpile(qc, sim)
        result = sim.run(t, shots=1, memory=True).result()
        raw = result.get_memory()[0].replace(' ', '')
        # Qiskit stores qubit 0 at the rightmost position — reverse to align
        bits.extend([int(b) for b in reversed(raw)])
    return bits[:n]


# Quick sanity check
test_bits = quantum_random_bits(200)
ones  = sum(test_bits)
zeros = len(test_bits) - ones
print(f"Quantum RNG check (200 bits): {zeros} zeros, {ones} ones")
print("Sample:", test_bits[:20])

Quantum RNG check (200 bits): 94 zeros, 106 ones
Sample: [0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1]


## Alice: Encoding

Plain protocol.

| Bit | Basis 0 (Z) | Basis 1 (X) |
|-----|------------|-------------|
| 0   | $|0\rangle$ | $|{+}\rangle$ |
| 1   | $|1\rangle$ | $|{-}\rangle$ |

In [3]:
# ALICE

def alice_encode(bit: int, basis: int) -> QuantumCircuit:
    """
    Alice encodes a classical bit into a qubit.

    basis=0 (Z / Rectilinear): |0> for bit=0, |1> for bit=1
    basis=1 (X / Diagonal):    |+> for bit=0, |-> for bit=1

    Returns a QuantumCircuit representing the prepared qubit (no measurement).
    """
    qc = QuantumCircuit(1)            # Start in |0>
    if bit == 1:
        qc.x(0)                       # Flip to |1>
    if basis == 1:
        qc.h(0)                       # Rotate to diagonal basis
    return qc

## Eve: Intercept-and-Resend Attack

Eve intercepts every qubit on the channel. She:
1. Measures the qubit in a randomly chosen basis (quantum measurement is irreversible).
2. Records her measured bit value.
3. Prepares a **fresh** qubit encoding her measured value in her basis and forwards it to Bob.

Eve cannot copy the qubit (no-cloning theorem), so she must measure and re-prepare. This measurement disturbs the qubit whenever Eve's basis differs from Alice's.

In [4]:
# EVE

def eve_intercept_resend(encoding_qc: QuantumCircuit,
                         eve_basis: int) -> tuple[int, QuantumCircuit]:
    """
    Eve intercepts Alice's qubit and performs an intercept-and-resend attack.

    1. Measures Alice's qubit in eve_basis (irreversibly collapses the state).
    2. Re-encodes the measured bit in the same basis (a fresh qubit).
    3. Returns the measured bit and the new qubit circuit to forward to Bob.

    Due to the no-cloning theorem, Eve cannot copy the qubit.
    When her basis differs from Alice's, she introduces disturbance.

    Parameters
    ----------
    encoding_qc : QuantumCircuit  — Alice's prepared qubit
    eve_basis   : int             — Eve's randomly chosen basis (0=Z, 1=X)

    Returns
    -------
    eve_bit   : int              — The bit Eve measured
    new_qc    : QuantumCircuit   — The fresh qubit Eve sends on to Bob
    """
    # Step 1: Eve measures Alice's qubit in her chosen basis
    qc = encoding_qc.copy()
    qc.add_register(ClassicalRegister(1))
    if eve_basis == 1:
        qc.h(0)                       # Rotate to X basis before measuring
    qc.measure(0, 0)
    t = transpile(qc, sim)
    result = sim.run(t, shots=1, memory=True).result()
    eve_bit = int(result.get_memory()[0].replace(' ', ''))

    # Step 2: Eve re-encodes her measured result as a fresh qubit
    new_qc = QuantumCircuit(1)
    if eve_bit == 1:
        new_qc.x(0)
    if eve_basis == 1:
        new_qc.h(0)

    return eve_bit, new_qc

## Bob: Measurement

In [5]:
# BOB

def bob_measure(encoding_qc: QuantumCircuit, basis: int) -> int:
    """
    Bob measures the incoming qubit (which may have been tampered by Eve).

    basis=0 (Z / Rectilinear): measure directly
    basis=1 (X / Diagonal):    apply H then measure

    Returns the classical bit result (0 or 1).
    """
    qc = encoding_qc.copy()
    qc.add_register(ClassicalRegister(1))
    if basis == 1:
        qc.h(0)                       # Rotate back from diagonal basis
    qc.measure(0, 0)
    t = transpile(qc, sim)
    result = sim.run(t, shots=1, memory=True).result()
    return int(result.get_memory()[0].replace(' ', ''))

## Running the BB84 Protocol With Eve

Full sequence:

$$\text{Alice} \xrightarrow{\text{qubit}} \text{Eve} \xrightarrow{\text{fresh qubit}} \text{Bob}$$

Alice and Bob are unaware of Eve's presence. They proceed with sifting and error-checking as normal.

In [6]:
# PROTOCOL PARAMETERS
N = 100        # Number of qubits transmitted
THRESHOLD = 0.11  # Error rate above which an attack is reported (11%)
                  # Without attacker: expected ~0%; with attacker: expected ~25%

print("=" * 60)
print("BB84 Protocol — With Attacker (Eve)")
print("=" * 60)
print(f"Transmitting N = {N} qubits\n")

BB84 Protocol — With Attacker (Eve)
Transmitting N = 100 qubits



In [7]:
# STEP 1: ALICE generates random bits and bases
print("[ALICE] Generating random bits and bases via quantum measurement...")
alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

print(f"[ALICE] Bits  (first 20): {alice_bits[:20]}")
print(f"[ALICE] Bases (first 20): {alice_bases[:20]}  (0=Z/Rectilinear, 1=X/Diagonal)")

[ALICE] Generating random bits and bases via quantum measurement...
[ALICE] Bits  (first 20): [1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1]
[ALICE] Bases (first 20): [1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0]  (0=Z/Rectilinear, 1=X/Diagonal)


In [8]:
# STEP 2: EVE generates random interception bases
print("[EVE] Generating random interception bases via quantum measurement...")
eve_bases = quantum_random_bits(N)

print(f"[EVE] Bases (first 20): {eve_bases[:20]}  (0=Z/Rectilinear, 1=X/Diagonal)")

[EVE] Generating random interception bases via quantum measurement...
[EVE] Bases (first 20): [0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1]  (0=Z/Rectilinear, 1=X/Diagonal)


In [9]:
# STEP 3: BOB generates random measurement bases
print("[BOB] Generating random measurement bases via quantum measurement...")
bob_bases = quantum_random_bits(N)

print(f"[BOB] Bases (first 20): {bob_bases[:20]}  (0=Z/Rectilinear, 1=X/Diagonal)")

[BOB] Generating random measurement bases via quantum measurement...
[BOB] Bases (first 20): [1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0]  (0=Z/Rectilinear, 1=X/Diagonal)


In [10]:
# STEP 4: QUANTUM CHANNEL (WITH EVE)
# Alice -> [Eve intercepts] -> Bob
print("[QUANTUM CHANNEL] Alice encodes; Eve intercepts and resends; Bob measures...")

eve_bits    = []    # What Eve measured (private information)
bob_results = []    # What Bob measured

for i in range(N):
    # Alice encodes qubit i and transmits
    qc_alice = alice_encode(alice_bits[i], alice_bases[i])

    # Eve intercepts: measures and re-sends a fresh qubit
    eve_bit, qc_eve = eve_intercept_resend(qc_alice, eve_bases[i])
    eve_bits.append(eve_bit)

    # Bob receives Eve's re-encoded qubit (not Alice's original)
    b = bob_measure(qc_eve, bob_bases[i])
    bob_results.append(b)

print(f"[EVE]  Measured bits (first 20): {eve_bits[:20]}")
print(f"[BOB]  Results       (first 20): {bob_results[:20]}")

[QUANTUM CHANNEL] Alice encodes; Eve intercepts and resends; Bob measures...
[EVE]  Measured bits (first 20): [1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
[BOB]  Results       (first 20): [1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1]


In [11]:
# STEP 5: SIFTING — Alice and Bob compare bases publicly
# Eve's bases are unknown to Alice and Bob at this point.
print("[PUBLIC CHANNEL] Alice and Bob compare bases (not bits)...")

sifted_indices = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
sifted_alice   = [alice_bits[i]  for i in sifted_indices]
sifted_bob     = [bob_results[i] for i in sifted_indices]

# track which sifted positions Eve also got right
eve_correct = [eve_bases[i] == alice_bases[i] for i in sifted_indices]

print(f"[SIFTING] Matching bases at {len(sifted_indices)}/{N} positions")
print(f"[ALICE] Sifted key (first 20): {sifted_alice[:20]}")
print(f"[BOB]   Sifted key (first 20): {sifted_bob[:20]}")
print(f"[EVE]   Correct basis (first 20): {eve_correct[:20]}")

[PUBLIC CHANNEL] Alice and Bob compare bases (not bits)...
[SIFTING] Matching bases at 40/100 positions
[ALICE] Sifted key (first 20): [1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1]
[BOB]   Sifted key (first 20): [1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1]
[EVE]   Correct basis (first 20): [False, False, True, False, False, True, True, False, False, True, True, True, True, False, False, True, True, True, False, True]


In [12]:
# STEP 6: ERROR CHECKING
# Alice and Bob compare a sample of their sifted bits over the public channel.
# Discrepancies reveal Eve's disturbance.
print("[PUBLIC CHANNEL] Comparing a sample of sifted key bits to check for errors...")

sample_size  = max(1, len(sifted_alice) // 4)   # Use 25% as check sample
sample_alice = sifted_alice[:sample_size]
sample_bob   = sifted_bob[:sample_size]

errors     = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / sample_size

print(f"\n[RESULT] Sample size:        {sample_size} bits")
print(f"[RESULT] Errors found:       {errors}")
print(f"[RESULT] Error rate:         {error_rate:.2%}")
print(f"[RESULT] Threshold:          {THRESHOLD:.2%}")
print(f"[THEORY] Expected error rate with full interception: ~25%")

if error_rate > THRESHOLD:
    print(f"\n[ALERT] Error rate {error_rate:.2%} exceeds threshold {THRESHOLD:.2%}.")
    print("           ATTACK DETECTED — eavesdropper present! Key exchange aborted.")
else:
    final_key = sifted_alice[sample_size:]
    print(f"\n[SUCCESS] No attack detected. Shared key established.")
    print(f"   Final key length: {len(final_key)} bits")
    print(f"   Final key (first 20): {final_key[:20]}")

[PUBLIC CHANNEL] Comparing a sample of sifted key bits to check for errors...

[RESULT] Sample size:        10 bits
[RESULT] Errors found:       2
[RESULT] Error rate:         20.00%
[RESULT] Threshold:          11.00%
[THEORY] Expected error rate with full interception: ~25%

[ALERT] Error rate 20.00% exceeds threshold 11.00%.
           ATTACK DETECTED — eavesdropper present! Key exchange aborted.


In [13]:
# ANALYSIS: Eve's information gain vs. detection
print("=" * 60)
print("Analysis: Eve's information gain vs. detection probability")
print("=" * 60)

# Among sifted positions (Alice's basis == Bob's basis), how often did Eve also use the correct basis?
eve_correct_count = sum(eve_correct)
eve_correct_rate  = eve_correct_count / len(sifted_indices) if sifted_indices else 0

# Errors caused specifically at positions where Eve guessed wrong
wrong_basis_errors = sum(
    sifted_alice[j] != sifted_bob[j]
    for j in range(len(sifted_indices))
    if not eve_correct[j]
)
wrong_basis_count = sum(1 for c in eve_correct if not c)

print(f"Sifted positions:          {len(sifted_indices)}")
print(f"Eve correct basis:         {eve_correct_count} ({eve_correct_rate:.1%}) — Eve learns bit correctly")
print(f"Eve wrong basis:           {wrong_basis_count} ({1-eve_correct_rate:.1%}) — qubit disturbed")
print(f"Errors at wrong-basis pos: {wrong_basis_errors}/{wrong_basis_count} "
      f"({'N/A' if wrong_basis_count == 0 else f'{wrong_basis_errors/wrong_basis_count:.1%}'})")
print()
print("Theoretical analysis:")
print("  Eve guesses basis correctly:  50% of the time (no disturbance)")
print("  Eve guesses wrong:            50% — causes error with prob 50%")
print("  Net error rate introduced:    25%")
print(f"  Observed error rate:         {error_rate:.2%}")

Analysis: Eve's information gain vs. detection probability
Sifted positions:          40
Eve correct basis:         21 (52.5%) — Eve learns bit correctly
Eve wrong basis:           19 (47.5%) — qubit disturbed
Errors at wrong-basis pos: 6/19 (31.6%)

Theoretical analysis:
  Eve guesses basis correctly:  50% of the time (no disturbance)
  Eve guesses wrong:            50% — causes error with prob 50%
  Net error rate introduced:    25%
  Observed error rate:         20.00%


## Comparing Attack vs. No-Attack Error Rates

Run a side-by-side comparison for clearer detection.

In [14]:
# SIDE-BY-SIDE COMPARISON
print("=" * 60)
print("Side-by-side: Protocol with vs. without Eve")
print("=" * 60)

# Run a fresh plain protocol for comparison
def run_plain(n):
    a_bits  = quantum_random_bits(n)
    a_bases = quantum_random_bits(n)
    b_bases = quantum_random_bits(n)
    b_res   = [bob_measure(alice_encode(a_bits[i], a_bases[i]), b_bases[i]) for i in range(n)]
    sifted  = [(a_bits[i], b_res[i]) for i in range(n) if a_bases[i] == b_bases[i]]
    sample  = max(1, len(sifted) // 4)
    errs    = sum(a != b for a, b in sifted[:sample])
    return errs / sample, len(sifted)

plain_rate, plain_sifted = run_plain(N)

print(f"{'Scenario':<30} {'Sifted bits':<15} {'Error rate':<15} {'Detected?':<10}")
print("-" * 70)
print(f"{'Without attacker':<30} {plain_sifted:<15} {plain_rate:<15.2%} {'No' if plain_rate <= THRESHOLD else 'Yes':<10}")
print(f"{'With Eve (full intercept)':<30} {len(sifted_indices):<15} {error_rate:<15.2%} {'Yes' if error_rate > THRESHOLD else 'No':<10}")
print()
print(f"Detection threshold: {THRESHOLD:.2%}")

Side-by-side: Protocol with vs. without Eve
Scenario                       Sifted bits     Error rate      Detected? 
----------------------------------------------------------------------
Without attacker               51              0.00%           No        
With Eve (full intercept)      40              20.00%          Yes       

Detection threshold: 11.00%


## Summary

| Party | Role | Action |
|-------|------|--------|
| Alice | Sender | Encodes bits into qubits using a randomly chosen basis |
| Eve   | Attacker | Intercepts each qubit, measures in a random basis, re-sends a fresh qubit |
| Bob   | Receiver | Measures each qubit in a randomly chosen basis |

### Why Eve Cannot Avoid Detection

The no-cloning theorem prevents Eve from copying qubits without disturbing them. She must measure, and measurement in the wrong basis collapses the quantum state, introducing unavoidable errors.

| Situation | Probability | Error introduced? |
|-----------|------------|------------------|
| Eve guesses basis correctly | 50% | No |
| Eve guesses wrong, Bob gets right answer | 25% | No |
| Eve guesses wrong, Bob gets wrong answer | 25% | Yes |

Expected error rate: 25%, far above the 11% threshold, so Eve is reliably detected.

If the error rate exceeds the threshold, Alice and Bob abort the key exchange and try again over a different channel. The security of BB84 rests entirely on the laws of quantum mechanics.